# CVPR GPU Worker\n\nColab Pro/Pro+ execution notebook for live CVPR demo outputs. Run this on a GPU runtime, then download `cvpr_gpu_results.json` into `source-code/learning/cvpr-colab-gpu-worker/_incoming/` for intake and promotion.\n\nThe first live path is open-vocabulary grounding with GroundingDINO-style zero-shot detection plus SigLIP image-text scoring. The deterministic fallback preserves the JSON contract when model weights are unavailable.\n

In [ ]:
import json, os, time\nfrom pathlib import Path\n\ntry:\n    import torch\n    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'\nexcept Exception:\n    accelerator = 'unknown'\n\nprint('accelerator:', accelerator)\n

In [ ]:
WORKER_JOBS = [
  {
    "id": "open-vocab-grounding",
    "title": "Open-vocabulary grounding GPU run",
    "bench": "cvpr-long-tail-grounding-bench",
    "page": "cvpr-long-tail-grounding-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "siglip-base-patch16-224",
      "grounding-dino-tiny",
      "sam-vit-b"
    ],
    "inputs": [
      "image",
      "text_query",
      "candidate_regions"
    ],
    "outputs": [
      "boxes",
      "region_scores",
      "embedding_scores",
      "localized_evidence"
    ],
    "gpuClass": "T4/L4/A100",
    "priority": 1
  },
  {
    "id": "restoration-fidelity",
    "title": "Restoration fidelity GPU run",
    "bench": "cvpr-restoration-fidelity-bench",
    "page": "cvpr-restoration-fidelity-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "swinir-lightweight",
      "real-esrgan-x2"
    ],
    "inputs": [
      "degraded_image",
      "degradation_controls"
    ],
    "outputs": [
      "restored_image",
      "artifact_map",
      "downstream_score"
    ],
    "gpuClass": "T4/L4/A100",
    "priority": 2
  },
  {
    "id": "adversarial-provenance",
    "title": "Adversarial provenance GPU run",
    "bench": "cvpr-adversarial-provenance-bench",
    "page": "cvpr-adversarial-provenance-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "watermark-detector",
      "clip-perturbation-probe"
    ],
    "inputs": [
      "image",
      "attack_controls",
      "watermark_controls"
    ],
    "outputs": [
      "provenance_confidence",
      "attack_heatmap",
      "leakage_risk"
    ],
    "gpuClass": "T4/L4/A100",
    "priority": 3
  },
  {
    "id": "temporal-rollout",
    "title": "Temporal rollout GPU run",
    "bench": "cvpr-temporal-rollout-bench",
    "page": "cvpr-temporal-rollout-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "video-feature-tracker",
      "raft-lite",
      "world-rollout-probe"
    ],
    "inputs": [
      "video_clip",
      "tracking_controls"
    ],
    "outputs": [
      "identity_tracks",
      "contact_events",
      "drift_curve"
    ],
    "gpuClass": "L4/A100",
    "priority": 4
  },
  {
    "id": "clinical-shift",
    "title": "Clinical shift validation GPU run",
    "bench": "cvpr-clinical-shift-bench",
    "page": "cvpr-clinical-shift-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "dicom-embedding-shift-probe",
      "temperature-calibration-head",
      "uncertainty-triage-head"
    ],
    "inputs": [
      "medical_image_batch",
      "site_metadata",
      "review_controls"
    ],
    "outputs": [
      "domain_embeddings",
      "calibration_curve",
      "triage_scores",
      "clinical_evidence"
    ],
    "gpuClass": "T4/L4/A100",
    "priority": 5
  },
  {
    "id": "compute-serving",
    "title": "Compute constrained serving GPU run",
    "bench": "cvpr-compute-serving-bench",
    "page": "cvpr-compute-serving-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "quantized-vision-encoder",
      "student-router",
      "latency-profiler"
    ],
    "inputs": [
      "image_batch",
      "serving_controls",
      "escalation_policy"
    ],
    "outputs": [
      "latency_profile",
      "quality_floor",
      "routing_trace",
      "retained_evidence"
    ],
    "gpuClass": "T4/L4/A100",
    "priority": 6
  },
  {
    "id": "constraint-generation",
    "title": "Constraint preserving generation GPU run",
    "bench": "cvpr-constraint-generation-bench",
    "page": "cvpr-constraint-generation-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "layout-controlnet",
      "identity-embedding-lock",
      "preference-reward-probe"
    ],
    "inputs": [
      "source_image",
      "edit_prompt",
      "constraint_controls"
    ],
    "outputs": [
      "edited_image",
      "layout_mask",
      "identity_embedding_delta",
      "reward_trace"
    ],
    "gpuClass": "L4/A100",
    "priority": 7
  },
  {
    "id": "driving-safety",
    "title": "Driving safety closed-loop GPU run",
    "bench": "cvpr-driving-safety-bench",
    "page": "cvpr-driving-safety-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "vla-scene-grounder",
      "ttc-risk-head",
      "safety-rule-monitor"
    ],
    "inputs": [
      "driving_clip",
      "hazard_controls",
      "action_confidence"
    ],
    "outputs": [
      "scene_grounding_map",
      "time_to_collision",
      "risk_trace",
      "rule_violations"
    ],
    "gpuClass": "L4/A100",
    "priority": 8
  },
  {
    "id": "metric-geometry",
    "title": "Metric geometry GPU run",
    "bench": "cvpr-metric-geometry-bench",
    "page": "cvpr-metric-geometry-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "torch-pose-bundle-adjuster",
      "metric-scale-probe",
      "surface-consistency-head"
    ],
    "inputs": [
      "multi_view_images",
      "camera_controls",
      "scale_controls"
    ],
    "outputs": [
      "pose_graph",
      "scale_trace",
      "surface_residual_map",
      "topology_warnings"
    ],
    "gpuClass": "L4/A100",
    "priority": 9
  },
  {
    "id": "gaussian-splatting",
    "title": "Gaussian Splatting GPU run",
    "bench": "cvpr-gaussian-splatting-bench",
    "page": "cvpr-gaussian-splatting-bench.html",
    "runtimeModes": [
      "simulated",
      "cached-real",
      "live-colab"
    ],
    "models": [
      "torch-splat-renderer",
      "semantic-splat-attach",
      "provenance-trace-head"
    ],
    "inputs": [
      "scene_views",
      "splat_controls",
      "edit_controls"
    ],
    "outputs": [
      "novel_view_renders",
      "semantic_splat_map",
      "provenance_trace",
      "edit_leakage_report"
    ],
    "gpuClass": "L4/A100",
    "priority": 10
  }
]\nRUN_MANIFEST = {
  "runtimePlane": "google-colab-pro-plus",
  "controlPlane": "local-static-cvpr-site",
  "resultArtifact": "source-code/learning/cvpr-colab-gpu-worker/_results/cvpr_gpu_results.json",
  "liveExportArtifact": "source-code/learning/cvpr-colab-gpu-worker/_incoming/cvpr_gpu_results_live.json",
  "notebook": "notebooks/cvpr_gpu_worker.ipynb",
  "jobs": [
    {
      "jobId": "open-vocab-grounding",
      "bench": "cvpr-long-tail-grounding-bench",
      "page": "cvpr-long-tail-grounding-bench.html",
      "priority": 1,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "siglip-base-patch16-224",
        "grounding-dino-tiny",
        "sam-vit-b"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_long_tail_grounding_bench/registry.json",
      "resultFilter": {
        "jobId": "open-vocab-grounding",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "restoration-fidelity",
      "bench": "cvpr-restoration-fidelity-bench",
      "page": "cvpr-restoration-fidelity-bench.html",
      "priority": 2,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "swinir-lightweight",
        "real-esrgan-x2"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_restoration_fidelity_bench/registry.json",
      "resultFilter": {
        "jobId": "restoration-fidelity",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "adversarial-provenance",
      "bench": "cvpr-adversarial-provenance-bench",
      "page": "cvpr-adversarial-provenance-bench.html",
      "priority": 3,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "watermark-detector",
        "clip-perturbation-probe"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_adversarial_provenance_bench/registry.json",
      "resultFilter": {
        "jobId": "adversarial-provenance",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "temporal-rollout",
      "bench": "cvpr-temporal-rollout-bench",
      "page": "cvpr-temporal-rollout-bench.html",
      "priority": 4,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "video-feature-tracker",
        "raft-lite",
        "world-rollout-probe"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_temporal_rollout_bench/registry.json",
      "resultFilter": {
        "jobId": "temporal-rollout",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "clinical-shift",
      "bench": "cvpr-clinical-shift-bench",
      "page": "cvpr-clinical-shift-bench.html",
      "priority": 5,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "dicom-embedding-shift-probe",
        "temperature-calibration-head",
        "uncertainty-triage-head"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_clinical_shift_bench/registry.json",
      "resultFilter": {
        "jobId": "clinical-shift",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "compute-serving",
      "bench": "cvpr-compute-serving-bench",
      "page": "cvpr-compute-serving-bench.html",
      "priority": 6,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "quantized-vision-encoder",
        "student-router",
        "latency-profiler"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_compute_serving_bench/registry.json",
      "resultFilter": {
        "jobId": "compute-serving",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "constraint-generation",
      "bench": "cvpr-constraint-generation-bench",
      "page": "cvpr-constraint-generation-bench.html",
      "priority": 7,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "layout-controlnet",
        "identity-embedding-lock",
        "preference-reward-probe"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_constraint_generation_bench/registry.json",
      "resultFilter": {
        "jobId": "constraint-generation",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "driving-safety",
      "bench": "cvpr-driving-safety-bench",
      "page": "cvpr-driving-safety-bench.html",
      "priority": 8,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "vla-scene-grounder",
        "ttc-risk-head",
        "safety-rule-monitor"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_driving_safety_bench/registry.json",
      "resultFilter": {
        "jobId": "driving-safety",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "metric-geometry",
      "bench": "cvpr-metric-geometry-bench",
      "page": "cvpr-metric-geometry-bench.html",
      "priority": 9,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "torch-pose-bundle-adjuster",
        "metric-scale-probe",
        "surface-consistency-head"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_metric_geometry_bench/registry.json",
      "resultFilter": {
        "jobId": "metric-geometry",
        "mode": "cached-real"
      }
    },
    {
      "jobId": "gaussian-splatting",
      "bench": "cvpr-gaussian-splatting-bench",
      "page": "cvpr-gaussian-splatting-bench.html",
      "priority": 10,
      "runtimeModes": [
        "simulated",
        "cached-real",
        "live-colab"
      ],
      "models": [
        "torch-splat-renderer",
        "semantic-splat-attach",
        "provenance-trace-head"
      ],
      "expectedCases": 4,
      "importPath": "analysis/cvpr_gaussian_splatting_bench/registry.json",
      "resultFilter": {
        "jobId": "gaussian-splatting",
        "mode": "cached-real"
      }
    }
  ]
}\nprint('jobs:', [job['id'] for job in WORKER_JOBS])\nprint('expected live results:', sum(job['expectedCases'] for job in RUN_MANIFEST['jobs']))\n

In [ ]:
# Run this install cell in Colab before the live open-vocab job.\n# The next cell still has a deterministic fallback so schema validation can run without model downloads.\n%pip -q install 'transformers>=4.44' accelerate pillow sentencepiece protobuf\n

In [ ]:
GROUNDING_CASES = [
  {
    "id": "common-clean",
    "title": "Common clean object",
    "controls": {
      "queryRarity": 18,
      "distractorOverlap": 16,
      "boxAmbiguity": 18,
      "evidenceThreshold": 54
    },
    "expectedMetrics": {
      "proposalRecall": 82.8,
      "textRegionScore": 84.7,
      "longTailRecall": 71.7,
      "localizedEvidence": 88.9,
      "unsupportedRisk": 8.3,
      "readiness": 84.7
    },
    "asset": "fixtures/open-vocab/common-clean.png"
  },
  {
    "id": "rare-visible",
    "title": "Rare visible object",
    "controls": {
      "queryRarity": 66,
      "distractorOverlap": 12,
      "boxAmbiguity": 34,
      "evidenceThreshold": 62
    },
    "expectedMetrics": {
      "proposalRecall": 76.8,
      "textRegionScore": 85.0,
      "longTailRecall": 76.9,
      "localizedEvidence": 87.7,
      "unsupportedRisk": 16.3,
      "readiness": 83.9
    },
    "asset": "fixtures/open-vocab/rare-visible.png"
  },
  {
    "id": "rare-distractors",
    "title": "Rare object with distractors",
    "controls": {
      "queryRarity": 78,
      "distractorOverlap": 28,
      "boxAmbiguity": 28,
      "evidenceThreshold": 76
    },
    "expectedMetrics": {
      "proposalRecall": 76.0,
      "textRegionScore": 83.6,
      "longTailRecall": 81.4,
      "localizedEvidence": 87.1,
      "unsupportedRisk": 19.0,
      "readiness": 83.8
    },
    "asset": "fixtures/open-vocab/rare-distractors.png"
  },
  {
    "id": "unsupported-query",
    "title": "Unsupported text query",
    "controls": {
      "queryRarity": 82,
      "distractorOverlap": 30,
      "boxAmbiguity": 32,
      "evidenceThreshold": 84
    },
    "expectedMetrics": {
      "proposalRecall": 75.3,
      "textRegionScore": 84.0,
      "longTailRecall": 82.1,
      "localizedEvidence": 87.1,
      "unsupportedRisk": 20.1,
      "readiness": 83.8
    },
    "asset": "fixtures/open-vocab/unsupported-query.png"
  }
]\n\ndef _clamp(value, lo=0, hi=100):\n    return max(lo, min(hi, float(value)))\n\ndef make_synthetic_grounding_image(case, size=384):\n    from PIL import Image, ImageDraw\n    controls = case['controls']\n    img = Image.new('RGB', (size, size), (236, 241, 239))\n    draw = ImageDraw.Draw(img)\n    target = [int(size * 0.18), int(size * 0.22), int(size * 0.48), int(size * 0.50)]\n    shift = int(controls['distractorOverlap'] * 0.9)\n    distractor = [int(size * 0.55) - shift, int(size * 0.26), int(size * 0.82) - shift, int(size * 0.52)]\n    draw.rectangle(distractor, fill=(195, 119, 59), outline=(92, 74, 57), width=4)\n    draw.rectangle(target, fill=(35, 129, 141), outline=(10, 90, 98), width=5)\n    draw.text((target[0], max(2, target[1] - 18)), case['title'][:28], fill=(16, 23, 25))\n    return img\n\ndef load_open_vocab_models(device=None):\n    import torch\n    from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, SiglipModel, SiglipProcessor\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    detector_id = 'IDEA-Research/grounding-dino-tiny'\n    siglip_id = 'google/siglip-base-patch16-224'\n    detector_processor = AutoProcessor.from_pretrained(detector_id)\n    detector = AutoModelForZeroShotObjectDetection.from_pretrained(detector_id).to(device).eval()\n    siglip_processor = SiglipProcessor.from_pretrained(siglip_id)\n    siglip = SiglipModel.from_pretrained(siglip_id).to(device).eval()\n    return {'device': device, 'detectorId': detector_id, 'siglipId': siglip_id, 'detectorProcessor': detector_processor, 'detector': detector, 'siglipProcessor': siglip_processor, 'siglip': siglip}\n\ndef deterministic_open_vocab_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('proposalRecall', 'textRegionScore', 'longTailRecall', 'localizedEvidence', 'unsupportedRisk', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    hardness = (controls['queryRarity'] + controls['distractorOverlap'] + controls['boxAmbiguity']) / 3\n    localized = _clamp(82 - hardness * 0.32 + controls['evidenceThreshold'] * 0.08)\n    unsupported = _clamp(hardness * 0.48 - controls['evidenceThreshold'] * 0.12)\n    readiness = _clamp(localized * 0.58 + (100 - unsupported) * 0.42)\n    return {'proposalRecall': round(localized, 1), 'textRegionScore': round(localized, 1), 'longTailRecall': round(localized, 1), 'localizedEvidence': round(localized, 1), 'unsupportedRisk': round(unsupported, 1), 'readiness': round(readiness, 1)}\n\ndef run_open_vocab_grounding_case(case, models=None, require_real_models=False):\n    import torch\n    image = make_synthetic_grounding_image(case)\n    query = case['title'].lower()\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_open_vocab_metrics(case)\n        outputs = {'boxes': [], 'regionScores': {'target': metrics['textRegionScore'], 'longTail': metrics['longTailRecall']}, 'localizedEvidence': metrics['localizedEvidence']}\n        return metrics, outputs, 'deterministic-fallback'\n    device = models['device']\n    detector_inputs = models['detectorProcessor'](images=image, text=query, return_tensors='pt').to(device)\n    with torch.no_grad():\n        detector_outputs = models['detector'](**detector_inputs)\n    processed = models['detectorProcessor'].post_process_grounded_object_detection(detector_outputs, detector_inputs.input_ids, box_threshold=0.25, text_threshold=0.20, target_sizes=[image.size[::-1]])[0]\n    boxes = []\n    for box, score, label in zip(processed.get('boxes', []), processed.get('scores', []), processed.get('labels', [])):\n        x0, y0, x1, y1 = [float(v) for v in box.tolist()]\n        boxes.append({'label': str(label), 'xywh': [round(x0 / image.width, 3), round(y0 / image.height, 3), round((x1 - x0) / image.width, 3), round((y1 - y0) / image.height, 3)], 'score': round(float(score), 3)})\n    siglip_inputs = models['siglipProcessor'](text=[query], images=image, padding='max_length', return_tensors='pt').to(device)\n    with torch.no_grad():\n        siglip_outputs = models['siglip'](**siglip_inputs)\n    embedding_score = torch.sigmoid(siglip_outputs.logits_per_image[0, 0]).item() * 100\n    proposal = _clamp(max([box['score'] for box in boxes], default=0.0) * 100)\n    controls = case['controls']\n    text_region = _clamp(embedding_score * 0.72 + proposal * 0.28)\n    long_tail = _clamp(text_region * 0.58 + controls['queryRarity'] * 0.16 + (100 - controls['boxAmbiguity']) * 0.26)\n    localized = _clamp(proposal * 0.38 + text_region * 0.42 + controls['evidenceThreshold'] * 0.20)\n    unsupported = _clamp((100 - localized) * 0.42 + controls['distractorOverlap'] * 0.30 + controls['boxAmbiguity'] * 0.22 - controls['evidenceThreshold'] * 0.16)\n    readiness = _clamp(localized * 0.34 + text_region * 0.24 + long_tail * 0.22 + (100 - unsupported) * 0.20)\n    metrics = {'proposalRecall': round(proposal, 1), 'textRegionScore': round(text_region, 1), 'longTailRecall': round(long_tail, 1), 'localizedEvidence': round(localized, 1), 'unsupportedRisk': round(unsupported, 1), 'readiness': round(readiness, 1)}\n    outputs = {'boxes': boxes, 'regionScores': {'target': metrics['textRegionScore'], 'longTail': metrics['longTailRecall']}, 'embeddingScore': round(embedding_score, 1), 'localizedEvidence': metrics['localizedEvidence']}\n    return metrics, outputs, 'transformers-grounding-dino-siglip'\n\ndef run_open_vocab_grounding_batch(cases, require_real_models=False):\n    try:\n        models = load_open_vocab_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('model load failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_open_vocab_grounding_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'open-vocab-grounding', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'embedding': 'siglip-base-patch16-224', 'detector': 'grounding-dino-tiny', 'segmenter': 'sam-vit-b'}, 'inputs': {'textQuery': case['title'].lower(), 'controls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-long-tail-grounding-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when you want the notebook to fail instead of falling back.\nopen_vocab_results = run_open_vocab_grounding_batch(GROUNDING_CASES, require_real_models=False)\nPath('cvpr_gpu_results.json').write_text(json.dumps(open_vocab_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(open_vocab_results))\nprint(json.dumps(open_vocab_results[0], indent=2)[:1200])\n

In [ ]:
RESTORATION_CASES = [
  {
    "id": "mild-noise",
    "title": "Mild sensor noise",
    "controls": {
      "blur": 18,
      "noise": 24,
      "compression": 18,
      "lowLight": 20,
      "hallucinationPenalty": 36
    },
    "expectedMetrics": {
      "degradationLoad": 21.6,
      "diagnosisConfidence": 83.8,
      "fidelityScore": 82.2,
      "artifactRisk": 24.7,
      "downstreamUtility": 85.3,
      "fabricatedDetailRisk": 25.8,
      "readiness": 82.0
    },
    "asset": "fixtures/restoration/mild-noise.png"
  },
  {
    "id": "compressed-low-light",
    "title": "Compressed low-light image",
    "controls": {
      "blur": 32,
      "noise": 38,
      "compression": 54,
      "lowLight": 64,
      "hallucinationPenalty": 16
    },
    "expectedMetrics": {
      "degradationLoad": 43.6,
      "diagnosisConfidence": 75.7,
      "fidelityScore": 80.3,
      "artifactRisk": 37.4,
      "downstreamUtility": 80.9,
      "fabricatedDetailRisk": 29.4,
      "readiness": 77.7
    },
    "asset": "fixtures/restoration/compressed-low-light.png"
  },
  {
    "id": "motion-blur-task",
    "title": "Motion blur task frame",
    "controls": {
      "blur": 64,
      "noise": 36,
      "compression": 38,
      "lowLight": 36,
      "hallucinationPenalty": 16
    },
    "expectedMetrics": {
      "degradationLoad": 41.2,
      "diagnosisConfidence": 74.2,
      "fidelityScore": 79.0,
      "artifactRisk": 30.4,
      "downstreamUtility": 81.5,
      "fabricatedDetailRisk": 26.5,
      "readiness": 77.7
    },
    "asset": "fixtures/restoration/motion-blur-task.png"
  },
  {
    "id": "over-restored-detail",
    "title": "Over-restored fine detail",
    "controls": {
      "blur": 48,
      "noise": 54,
      "compression": 38,
      "lowLight": 56,
      "hallucinationPenalty": 18
    },
    "expectedMetrics": {
      "degradationLoad": 45.9,
      "diagnosisConfidence": 75.2,
      "fidelityScore": 80.1,
      "artifactRisk": 35.1,
      "downstreamUtility": 81.2,
      "fabricatedDetailRisk": 28.6,
      "readiness": 77.8
    },
    "asset": "fixtures/restoration/over-restored-detail.png"
  }
]\n\ndef make_degraded_restoration_image(case, size=384):\n    from PIL import Image, ImageDraw, ImageFilter, ImageEnhance\n    controls = case['controls']\n    img = Image.new('RGB', (size, size), (222, 229, 225))\n    draw = ImageDraw.Draw(img)\n    for i in range(12):\n        x = int((i * 37) % size)\n        color = (40 + i * 11, 110 + i * 5, 132 + i * 3)\n        draw.rectangle([x, 24 + i * 18, min(size - 1, x + 92), min(size - 1, 88 + i * 18)], fill=color)\n    draw.text((18, 18), case['title'][:30], fill=(15, 22, 24))\n    img = img.filter(ImageFilter.GaussianBlur(radius=max(0.1, controls['blur'] / 45)))\n    img = ImageEnhance.Brightness(img).enhance(max(0.35, 1 - controls['lowLight'] / 140))\n    img = ImageEnhance.Contrast(img).enhance(max(0.45, 1 - controls['compression'] / 180))\n    return img\n\ndef load_restoration_models(device=None):\n    import torch\n    from transformers import AutoImageProcessor, Swin2SRForImageSuperResolution\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    model_id = 'caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr'\n    processor = AutoImageProcessor.from_pretrained(model_id)\n    model = Swin2SRForImageSuperResolution.from_pretrained(model_id).to(device).eval()\n    return {'device': device, 'modelId': model_id, 'processor': processor, 'model': model}\n\ndef deterministic_restoration_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('degradationLoad', 'diagnosisConfidence', 'fidelityScore', 'artifactRisk', 'downstreamUtility', 'fabricatedDetailRisk', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    load = _clamp(controls['blur'] * 0.24 + controls['noise'] * 0.22 + controls['compression'] * 0.22 + controls['lowLight'] * 0.22 + controls['hallucinationPenalty'] * 0.10)\n    fidelity = _clamp(88 - load * 0.38 - controls['hallucinationPenalty'] * 0.22)\n    downstream = _clamp(72 + fidelity * 0.22 - load * 0.18)\n    fabricated = _clamp(controls['hallucinationPenalty'] * 0.55 + (100 - fidelity) * 0.22)\n    readiness = _clamp(fidelity * 0.34 + downstream * 0.34 + (100 - fabricated) * 0.32)\n    return {'degradationLoad': round(load, 1), 'diagnosisConfidence': round(100 - load, 1), 'fidelityScore': round(fidelity, 1), 'artifactRisk': round(fabricated, 1), 'downstreamUtility': round(downstream, 1), 'fabricatedDetailRisk': round(fabricated, 1), 'readiness': round(readiness, 1)}\n\ndef _image_delta_metric(before, after):\n    import numpy as np\n    before = before.resize((128, 128))\n    after = after.resize((128, 128))\n    a = np.asarray(before).astype('float32')\n    b = np.asarray(after).astype('float32')\n    delta = np.mean(np.abs(a - b)) / 255 * 100\n    return float(delta)\n\ndef run_restoration_fidelity_case(case, models=None, require_real_models=False):\n    from PIL import ImageFilter, ImageEnhance\n    import torch\n    image = make_degraded_restoration_image(case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_restoration_metrics(case)\n        restored = ImageEnhance.Sharpness(image.filter(ImageFilter.SHARPEN)).enhance(1.6)\n        execution = 'deterministic-fallback'\n    else:\n        inputs = models['processor'](image, return_tensors='pt').to(models['device'])\n        with torch.no_grad():\n            outputs = models['model'](**inputs)\n        tensor = outputs.reconstruction.squeeze().float().cpu().clamp(0, 1)\n        restored = models['processor'].post_process_image(tensor) if hasattr(models['processor'], 'post_process_image') else None\n        if restored is None:\n            import numpy as np\n            arr = (tensor.permute(1, 2, 0).numpy() * 255).astype('uint8')\n            from PIL import Image\n            restored = Image.fromarray(arr)\n        delta = _image_delta_metric(image, restored)\n        controls = case['controls']\n        fidelity = _clamp(86 - delta * 0.18 - controls['hallucinationPenalty'] * 0.18 + controls['noise'] * 0.06)\n        downstream = _clamp(74 + fidelity * 0.18 - controls['compression'] * 0.10 - controls['lowLight'] * 0.08)\n        fabricated = _clamp(controls['hallucinationPenalty'] * 0.48 + delta * 0.22 + controls['lowLight'] * 0.08)\n        load = _clamp(controls['blur'] * 0.24 + controls['noise'] * 0.22 + controls['compression'] * 0.22 + controls['lowLight'] * 0.22 + controls['hallucinationPenalty'] * 0.10)\n        readiness = _clamp(fidelity * 0.30 + downstream * 0.30 + (100 - fabricated) * 0.22 + (100 - load) * 0.18)\n        metrics = {'degradationLoad': round(load, 1), 'diagnosisConfidence': round(100 - load, 1), 'fidelityScore': round(fidelity, 1), 'artifactRisk': round(fabricated, 1), 'downstreamUtility': round(downstream, 1), 'fabricatedDetailRisk': round(fabricated, 1), 'readiness': round(readiness, 1)}\n        execution = 'transformers-swin2sr-restoration'\n    outputs = {'restoredImage': f"fixtures/restoration/{case['id']}-restored.png", 'artifactMap': f"fixtures/restoration/{case['id']}-artifact-map.png", 'downstreamScore': metrics['downstreamUtility'], 'fidelityScore': metrics['fidelityScore']}\n    return metrics, outputs, execution\n\ndef run_restoration_fidelity_batch(cases, require_real_models=False):\n    try:\n        models = load_restoration_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('restoration model load failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_restoration_fidelity_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'restoration-fidelity', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'restorer': 'swin2sr-realworld-sr-x4', 'artifactProbe': 'delta-artifact-map'}, 'inputs': {'degradationControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': {'readiness': metrics['readiness'], 'downstreamUtility': metrics['downstreamUtility'], 'fabricatedDetailRisk': metrics['fabricatedDetailRisk'], 'fidelityScore': metrics['fidelityScore']}, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-restoration-fidelity-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing restoration release evidence.\nrestoration_results = run_restoration_fidelity_batch(RESTORATION_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(restoration_results[0], indent=2)[:1200])\n

In [ ]:
ADVERSARIAL_CASES = [
  {
    "id": "clean-camera",
    "title": "Clean camera image",
    "controls": {
      "attackStrength": 0,
      "generationSource": 56,
      "watermarkVisibility": 100,
      "unlearningProbe": 0
    },
    "expectedMetrics": {
      "attackCoverage": 44.7,
      "provenanceConfidence": 86.4,
      "leakageRisk": 9.0,
      "evidence": 82.5,
      "risk": 17.9,
      "readiness": 79.7
    },
    "asset": "fixtures/adversarial/clean-camera.png"
  },
  {
    "id": "edited-social-post",
    "title": "Edited social post",
    "controls": {
      "attackStrength": 0,
      "generationSource": 64,
      "watermarkVisibility": 98,
      "unlearningProbe": 10
    },
    "expectedMetrics": {
      "attackCoverage": 47.2,
      "provenanceConfidence": 87.9,
      "leakageRisk": 14.8,
      "evidence": 82.9,
      "risk": 21.0,
      "readiness": 79.8
    },
    "asset": "fixtures/adversarial/edited-social-post.png"
  },
  {
    "id": "synthetic-watermarked",
    "title": "Synthetic watermarked media",
    "controls": {
      "attackStrength": 0,
      "generationSource": 84,
      "watermarkVisibility": 94,
      "unlearningProbe": 44
    },
    "expectedMetrics": {
      "attackCoverage": 54.4,
      "provenanceConfidence": 92.0,
      "leakageRisk": 32.9,
      "evidence": 83.6,
      "risk": 29.9,
      "readiness": 79.9
    },
    "asset": "fixtures/adversarial/synthetic-watermarked.png"
  },
  {
    "id": "adaptive-attack",
    "title": "Adaptive provenance attack",
    "controls": {
      "attackStrength": 12,
      "generationSource": 89,
      "watermarkVisibility": 100,
      "unlearningProbe": 0
    },
    "expectedMetrics": {
      "attackCoverage": 52.6,
      "provenanceConfidence": 93.8,
      "leakageRisk": 17.4,
      "evidence": 85.3,
      "risk": 30.3,
      "readiness": 80.8
    },
    "asset": "fixtures/adversarial/adaptive-attack.png"
  }
]\n\ndef make_adversarial_provenance_image(case, size=384):\n    from PIL import Image, ImageDraw, ImageFilter, ImageEnhance\n    controls = case['controls']\n    base = (232, 235, 232) if controls['generationSource'] < 50 else (218, 207, 226)\n    img = Image.new('RGB', (size, size), base)\n    draw = ImageDraw.Draw(img)\n    for i in range(10):\n        x = int((i * 43 + controls['generationSource']) % size)\n        y = int((i * 31 + controls['attackStrength']) % size)\n        draw.ellipse([x, y, min(size, x + 82), min(size, y + 58)], fill=(70 + i * 9, 95 + i * 8, 128 + i * 6))\n    if controls['watermarkVisibility'] > 20:\n        mark = f"WM {controls['watermarkVisibility']}"\n        draw.rectangle([size - 138, size - 58, size - 12, size - 18], outline=(15, 80, 88), width=3)\n        draw.text((size - 126, size - 47), mark, fill=(15, 80, 88))\n    if controls['attackStrength'] > 45:\n        img = ImageEnhance.Contrast(img).enhance(1 + controls['attackStrength'] / 180)\n        img = img.filter(ImageFilter.GaussianBlur(radius=controls['attackStrength'] / 95))\n    draw.text((18, 18), case['title'][:30], fill=(14, 22, 24))\n    return img\n\ndef load_adversarial_models(device=None):\n    import torch\n    from transformers import CLIPModel, CLIPProcessor\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    model_id = 'openai/clip-vit-base-patch32'\n    processor = CLIPProcessor.from_pretrained(model_id)\n    model = CLIPModel.from_pretrained(model_id).to(device).eval()\n    return {'device': device, 'modelId': model_id, 'processor': processor, 'model': model}\n\ndef deterministic_adversarial_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('attackCoverage', 'provenanceConfidence', 'leakageRisk', 'evidence', 'risk', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    coverage = _clamp(controls['attackStrength'] * 0.35 + controls['generationSource'] * 0.22 + (100 - controls['watermarkVisibility']) * 0.18 + controls['unlearningProbe'] * 0.25)\n    confidence = _clamp(controls['generationSource'] * 0.30 + controls['watermarkVisibility'] * 0.42 + (100 - controls['attackStrength']) * 0.18)\n    leakage = _clamp(controls['unlearningProbe'] * 0.42 + controls['attackStrength'] * 0.26 + controls['generationSource'] * 0.16 + (100 - controls['watermarkVisibility']) * 0.16)\n    evidence = _clamp(confidence * 0.36 + coverage * 0.18 + (100 - leakage) * 0.18 + 18)\n    risk = _clamp(controls['attackStrength'] * 0.34 + controls['generationSource'] * 0.20 + leakage * 0.28 + (100 - evidence) * 0.24)\n    readiness = _clamp(evidence * 0.42 + confidence * 0.26 + (100 - risk) * 0.22 + coverage * 0.10)\n    return {'attackCoverage': round(coverage, 1), 'provenanceConfidence': round(confidence, 1), 'leakageRisk': round(leakage, 1), 'evidence': round(evidence, 1), 'risk': round(risk, 1), 'readiness': round(readiness, 1)}\n\ndef run_adversarial_provenance_case(case, models=None, require_real_models=False):\n    import torch\n    image = make_adversarial_provenance_image(case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_adversarial_metrics(case)\n        outputs = {'provenanceConfidence': metrics['provenanceConfidence'], 'attackHeatmap': f"fixtures/adversarial/{case['id']}-attack-heatmap.png", 'leakageRisk': metrics['leakageRisk'], 'evidence': metrics['evidence']}\n        return metrics, outputs, 'deterministic-fallback'\n    prompts = ['a natural camera photograph', 'a synthetic generated image', 'an image with a visible watermark', 'an adversarially perturbed image']\n    inputs = models['processor'](text=prompts, images=image, return_tensors='pt', padding=True).to(models['device'])\n    with torch.no_grad():\n        logits = models['model'](**inputs).logits_per_image[0]\n    probs = torch.softmax(logits, dim=0).detach().cpu().tolist()\n    natural, synthetic, watermark, attacked = [float(v) for v in probs]\n    controls = case['controls']\n    coverage = _clamp(attacked * 100 * 0.55 + controls['attackStrength'] * 0.30 + (100 - controls['watermarkVisibility']) * 0.15)\n    confidence = _clamp(synthetic * 100 * 0.34 + watermark * 100 * 0.34 + controls['watermarkVisibility'] * 0.22 + (100 - controls['attackStrength']) * 0.10)\n    leakage = _clamp(controls['unlearningProbe'] * 0.42 + controls['attackStrength'] * 0.22 + synthetic * 100 * 0.18 + (100 - controls['watermarkVisibility']) * 0.18)\n    evidence = _clamp(confidence * 0.34 + coverage * 0.18 + (100 - leakage) * 0.16 + watermark * 100 * 0.20 + natural * 100 * 0.12)\n    risk = _clamp(controls['attackStrength'] * 0.32 + controls['generationSource'] * 0.18 + leakage * 0.30 + (100 - evidence) * 0.20)\n    readiness = _clamp(evidence * 0.42 + confidence * 0.26 + (100 - risk) * 0.22 + coverage * 0.10)\n    metrics = {'attackCoverage': round(coverage, 1), 'provenanceConfidence': round(confidence, 1), 'leakageRisk': round(leakage, 1), 'evidence': round(evidence, 1), 'risk': round(risk, 1), 'readiness': round(readiness, 1)}\n    outputs = {'provenanceConfidence': metrics['provenanceConfidence'], 'attackHeatmap': f"fixtures/adversarial/{case['id']}-attack-heatmap.png", 'leakageRisk': metrics['leakageRisk'], 'evidence': metrics['evidence'], 'clipProbe': {'natural': round(natural, 3), 'synthetic': round(synthetic, 3), 'watermark': round(watermark, 3), 'attacked': round(attacked, 3)}}\n    return metrics, outputs, 'transformers-clip-provenance-probe'\n\ndef run_adversarial_provenance_batch(cases, require_real_models=False):\n    try:\n        models = load_adversarial_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('adversarial provenance model load failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_adversarial_provenance_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'adversarial-provenance', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'detector': 'clip-vit-base-patch32-provenance-probe', 'probe': 'watermark-attack-prompt-bank'}, 'inputs': {'attackControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-adversarial-provenance-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing adversarial provenance release evidence.\nadversarial_results = run_adversarial_provenance_batch(ADVERSARIAL_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(adversarial_results[0], indent=2)[:1200])\n

In [ ]:
TEMPORAL_CASES = [
  {
    "id": "short-stable",
    "title": "Short stable rollout",
    "controls": {
      "rolloutLength": 24,
      "identityDensity": 28,
      "physicsViolations": 14,
      "memoryWindow": 72
    },
    "expectedMetrics": {
      "memoryLoad": 26.6,
      "identityStability": 80.6,
      "contactConsistency": 86.6,
      "rolloutPlausibility": 85.7,
      "drift": 21.0,
      "readiness": 83.3
    },
    "asset": "fixtures/temporal/short-stable.mp4"
  },
  {
    "id": "crowded-memory",
    "title": "Crowded identity memory",
    "controls": {
      "rolloutLength": 36,
      "identityDensity": 76,
      "physicsViolations": 26,
      "memoryWindow": 82
    },
    "expectedMetrics": {
      "memoryLoad": 43.8,
      "identityStability": 73.8,
      "contactConsistency": 79.4,
      "rolloutPlausibility": 80.0,
      "drift": 33.0,
      "readiness": 75.9
    },
    "asset": "fixtures/temporal/crowded-memory.mp4"
  },
  {
    "id": "contact-heavy",
    "title": "Contact-heavy prediction",
    "controls": {
      "rolloutLength": 56,
      "identityDensity": 52,
      "physicsViolations": 20,
      "memoryWindow": 82
    },
    "expectedMetrics": {
      "memoryLoad": 42.5,
      "identityStability": 74.0,
      "contactConsistency": 83.6,
      "rolloutPlausibility": 77.1,
      "drift": 34.3,
      "readiness": 76.2
    },
    "asset": "fixtures/temporal/contact-heavy.mp4"
  },
  {
    "id": "long-rollout-drift",
    "title": "Long rollout drift",
    "controls": {
      "rolloutLength": 66,
      "identityDensity": 68,
      "physicsViolations": 12,
      "memoryWindow": 92
    },
    "expectedMetrics": {
      "memoryLoad": 48.1,
      "identityStability": 72.7,
      "contactConsistency": 86.5,
      "rolloutPlausibility": 75.5,
      "drift": 36.1,
      "readiness": 75.9
    },
    "asset": "fixtures/temporal/long-rollout-drift.mp4"
  }
]\n\ndef make_temporal_rollout_frames(case, size=320):\n    from PIL import Image, ImageDraw\n    controls = case['controls']\n    frames = []\n    actors = max(2, int(controls['identityDensity'] / 22))\n    for step in range(2):\n        img = Image.new('RGB', (size, size), (231, 236, 233))\n        draw = ImageDraw.Draw(img)\n        draw.line([0, size * 0.72, size, size * 0.72], fill=(92, 105, 98), width=4)\n        for actor in range(actors):\n            x = 32 + actor * (size - 64) / max(1, actors) + step * (controls['rolloutLength'] / 8)\n            y = 70 + ((actor * 53 + controls['physicsViolations']) % 160)\n            if controls['physicsViolations'] > 55 and actor % 2 == 0:\n                y += step * controls['physicsViolations'] / 4\n            color = (28 + actor * 24, 118 + actor * 17, 137 + actor * 11)\n            draw.ellipse([x, y, x + 24, y + 24], fill=color, outline=(12, 44, 50), width=2)\n            draw.text((x, y - 14), f'id{actor}', fill=(15, 22, 24))\n        draw.text((14, 14), case['title'][:30], fill=(15, 22, 24))\n        frames.append(img)\n    return frames\n\ndef load_temporal_models(device=None):\n    import torch\n    from torchvision.models.optical_flow import Raft_Small_Weights, raft_small\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    weights = Raft_Small_Weights.DEFAULT\n    model = raft_small(weights=weights, progress=False).to(device).eval()\n    transforms = weights.transforms()\n    return {'device': device, 'modelId': 'torchvision-raft-small', 'model': model, 'transforms': transforms}\n\ndef deterministic_temporal_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('memoryLoad', 'identityStability', 'contactConsistency', 'rolloutPlausibility', 'drift', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    memory_load = _clamp(controls['rolloutLength'] * 0.34 + controls['identityDensity'] * 0.34 + (100 - controls['memoryWindow']) * 0.32)\n    identity = _clamp(82 - memory_load * 0.35 - controls['identityDensity'] * 0.12 + controls['memoryWindow'] * 0.18)\n    contact = _clamp(86 - controls['physicsViolations'] * 0.42 - controls['identityDensity'] * 0.08)\n    plausibility = _clamp(contact * 0.34 + identity * 0.28 + (100 - controls['rolloutLength']) * 0.22 + 12)\n    drift = _clamp(memory_load * 0.30 + (100 - identity) * 0.30 + controls['physicsViolations'] * 0.24 + controls['rolloutLength'] * 0.16)\n    readiness = _clamp(identity * 0.30 + contact * 0.28 + plausibility * 0.26 + (100 - drift) * 0.16)\n    return {'memoryLoad': round(memory_load, 1), 'identityStability': round(identity, 1), 'contactConsistency': round(contact, 1), 'rolloutPlausibility': round(plausibility, 1), 'drift': round(drift, 1), 'readiness': round(readiness, 1)}\n\ndef run_temporal_rollout_case(case, models=None, require_real_models=False):\n    import torch\n    frames = make_temporal_rollout_frames(case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_temporal_metrics(case)\n        execution = 'deterministic-fallback'\n    else:\n        import torchvision.transforms.functional as F\n        img1 = F.pil_to_tensor(frames[0]).float().unsqueeze(0).to(models['device']) / 255.0\n        img2 = F.pil_to_tensor(frames[1]).float().unsqueeze(0).to(models['device']) / 255.0\n        img1, img2 = models['transforms'](img1, img2)\n        with torch.no_grad():\n            flow = models['model'](img1, img2)[-1]\n        flow_mag = torch.linalg.vector_norm(flow, dim=1).mean().item()\n        controls = case['controls']\n        memory_load = _clamp(controls['rolloutLength'] * 0.34 + controls['identityDensity'] * 0.34 + (100 - controls['memoryWindow']) * 0.32)\n        identity = _clamp(88 - memory_load * 0.30 - flow_mag * 1.7 + controls['memoryWindow'] * 0.12)\n        contact = _clamp(84 - controls['physicsViolations'] * 0.38 - max(0, flow_mag - 6) * 1.6 + controls['memoryWindow'] * 0.08)\n        plausibility = _clamp(contact * 0.34 + identity * 0.26 + (100 - controls['rolloutLength']) * 0.18 + (100 - flow_mag) * 0.12)\n        drift = _clamp(memory_load * 0.28 + (100 - identity) * 0.32 + controls['physicsViolations'] * 0.22 + controls['rolloutLength'] * 0.14 + flow_mag * 0.35)\n        readiness = _clamp(identity * 0.30 + contact * 0.28 + plausibility * 0.26 + (100 - drift) * 0.16)\n        metrics = {'memoryLoad': round(memory_load, 1), 'identityStability': round(identity, 1), 'contactConsistency': round(contact, 1), 'rolloutPlausibility': round(plausibility, 1), 'drift': round(drift, 1), 'readiness': round(readiness, 1)}\n        execution = 'torchvision-raft-temporal-flow'\n    outputs = {'identityTracks': f"fixtures/temporal/{case['id']}-identity-tracks.json", 'contactEvents': f"fixtures/temporal/{case['id']}-contacts.json", 'driftCurve': [round(metrics['drift'] * point / 4, 1) for point in range(1, 5)], 'rolloutPlausibility': metrics['rolloutPlausibility']}\n    return metrics, outputs, execution\n\ndef run_temporal_rollout_batch(cases, require_real_models=False):\n    try:\n        models = load_temporal_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('temporal rollout model load failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_temporal_rollout_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'temporal-rollout', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'tracker': 'torchvision-raft-small', 'flow': 'raft-small', 'rolloutProbe': 'flow-drift-probe'}, 'inputs': {'trackingControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-temporal-rollout-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing temporal rollout release evidence.\ntemporal_results = run_temporal_rollout_batch(TEMPORAL_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results + temporal_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(temporal_results[0], indent=2)[:1200])\n

In [ ]:
CLINICAL_CASES = [
  {
    "id": "same-site-clean",
    "title": "Same-site clean validation",
    "controls": {
      "scannerShift": 16,
      "cohortMix": 22,
      "labelNoise": 8,
      "reviewThreshold": 62
    },
    "expectedMetrics": {
      "shiftLoad": 16.7,
      "calibration": 84.3,
      "domainEvidence": 89.1,
      "triageRate": 25.4,
      "residualRisk": 9.6,
      "clinicalEvidence": 90.3,
      "readiness": 88.5
    },
    "asset": "fixtures/clinical/same-site-clean.json"
  },
  {
    "id": "new-scanner",
    "title": "New scanner protocol",
    "controls": {
      "scannerShift": 58,
      "cohortMix": 34,
      "labelNoise": 16,
      "reviewThreshold": 68
    },
    "expectedMetrics": {
      "shiftLoad": 40.5,
      "calibration": 77.5,
      "domainEvidence": 78.6,
      "triageRate": 39.7,
      "residualRisk": 21.6,
      "clinicalEvidence": 84.8,
      "readiness": 80.5
    },
    "asset": "fixtures/clinical/new-scanner.json"
  },
  {
    "id": "external-hospital",
    "title": "External hospital cohort",
    "controls": {
      "scannerShift": 52,
      "cohortMix": 72,
      "labelNoise": 16,
      "reviewThreshold": 74
    },
    "expectedMetrics": {
      "shiftLoad": 52.4,
      "calibration": 75.4,
      "domainEvidence": 72.9,
      "triageRate": 47.0,
      "residualRisk": 26.4,
      "clinicalEvidence": 82.3,
      "readiness": 77.1
    },
    "asset": "fixtures/clinical/external-hospital.json"
  },
  {
    "id": "noisy-rare-cohort",
    "title": "Noisy rare cohort",
    "controls": {
      "scannerShift": 76,
      "cohortMix": 84,
      "labelNoise": 20,
      "reviewThreshold": 84
    },
    "expectedMetrics": {
      "shiftLoad": 67.8,
      "calibration": 72.4,
      "domainEvidence": 66.0,
      "triageRate": 57.0,
      "residualRisk": 33.5,
      "clinicalEvidence": 79.1,
      "readiness": 72.5
    },
    "asset": "fixtures/clinical/noisy-rare-cohort.json"
  }
]\n\ndef make_clinical_tensor(case, size=128):\n    import torch\n    controls = case['controls']\n    grid = torch.linspace(-1, 1, size)\n    yy, xx = torch.meshgrid(grid, grid, indexing='ij')\n    scanner = controls['scannerShift'] / 100\n    cohort = controls['cohortMix'] / 100\n    noise = controls['labelNoise'] / 100\n    blob = torch.exp(-((xx - scanner * 0.35) ** 2 + (yy + cohort * 0.25) ** 2) * (3.0 + cohort))\n    ring = torch.sin((xx * (4 + scanner * 4)) + (yy * (3 + cohort * 5))) * 0.12\n    artifact = torch.cos((xx + yy) * (8 + scanner * 6)) * scanner * 0.10\n    tensor = (blob + ring + artifact).clamp(0, 1)\n    if noise > 0:\n        gen = torch.Generator().manual_seed(sum(ord(c) for c in case['id']))\n        tensor = (tensor + torch.randn(tensor.shape, generator=gen) * noise * 0.10).clamp(0, 1)\n    return tensor.unsqueeze(0).unsqueeze(0)\n\ndef load_clinical_models(device=None):\n    import torch\n    import torch.nn as nn\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    model = nn.Sequential(\n        nn.Conv2d(1, 8, kernel_size=5, stride=2, padding=2), nn.ReLU(),\n        nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),\n        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),\n    ).to(device).eval()\n    return {'device': device, 'modelId': 'torch-clinical-shift-embedding-probe', 'model': model}\n\ndef deterministic_clinical_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('shiftLoad', 'calibration', 'domainEvidence', 'triageRate', 'residualRisk', 'clinicalEvidence', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    shift = _clamp(controls['scannerShift'] * 0.42 + controls['cohortMix'] * 0.38 + controls['labelNoise'] * 0.20)\n    calibration = _clamp(76 - shift * 0.28 - controls['labelNoise'] * 0.10 + controls['reviewThreshold'] * 0.22)\n    domain = _clamp(78 - controls['scannerShift'] * 0.28 - controls['cohortMix'] * 0.24)\n    triage = _clamp(shift * 0.45 + (100 - calibration) * 0.35 + controls['reviewThreshold'] * 0.20)\n    risk = _clamp(shift * 0.38 + controls['labelNoise'] * 0.26 + (100 - calibration) * 0.24 + (100 - domain) * 0.18 - triage * 0.18)\n    evidence = _clamp(domain * 0.38 + calibration * 0.26 + (100 - risk) * 0.18 + 16)\n    readiness = _clamp(evidence * 0.36 + calibration * 0.26 + domain * 0.22 + (100 - risk) * 0.16)\n    return {'shiftLoad': round(shift, 1), 'calibration': round(calibration, 1), 'domainEvidence': round(domain, 1), 'triageRate': round(triage, 1), 'residualRisk': round(risk, 1), 'clinicalEvidence': round(evidence, 1), 'readiness': round(readiness, 1)}\n\ndef run_clinical_shift_case(case, models=None, require_real_models=False):\n    import torch\n    tensor = make_clinical_tensor(case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_clinical_metrics(case)\n        execution = 'deterministic-fallback'\n    else:\n        tensor = tensor.to(models['device'])\n        with torch.no_grad():\n            emb = models['model'](tensor).flatten()\n        emb_norm = torch.linalg.vector_norm(emb).item()\n        emb_spread = torch.std(emb).item() if emb.numel() > 1 else 0.0\n        controls = case['controls']\n        shift = _clamp(controls['scannerShift'] * 0.42 + controls['cohortMix'] * 0.38 + controls['labelNoise'] * 0.20 + emb_spread * 8)\n        calibration = _clamp(82 - shift * 0.24 - controls['labelNoise'] * 0.16 + controls['reviewThreshold'] * 0.18 + emb_norm * 1.5)\n        domain = _clamp(76 - controls['scannerShift'] * 0.26 - controls['cohortMix'] * 0.20 + emb_norm * 2.0)\n        triage = _clamp(shift * 0.45 + (100 - calibration) * 0.35 + controls['reviewThreshold'] * 0.20)\n        risk = _clamp(shift * 0.38 + controls['labelNoise'] * 0.26 + (100 - calibration) * 0.24 + (100 - domain) * 0.18 - triage * 0.18)\n        evidence = _clamp(domain * 0.34 + calibration * 0.24 + (100 - risk) * 0.16 + emb_norm * 3.0 + 14)\n        readiness = _clamp(evidence * 0.36 + calibration * 0.26 + domain * 0.22 + (100 - risk) * 0.16)\n        metrics = {'shiftLoad': round(shift, 1), 'calibration': round(calibration, 1), 'domainEvidence': round(domain, 1), 'triageRate': round(triage, 1), 'residualRisk': round(risk, 1), 'clinicalEvidence': round(evidence, 1), 'readiness': round(readiness, 1)}\n        execution = 'torch-clinical-shift-embedding-probe'\n    outputs = {'domainEmbeddings': f"fixtures/clinical/{case['id']}-domain-embeddings.npy", 'calibrationCurve': f"fixtures/clinical/{case['id']}-calibration.json", 'triageScores': f"fixtures/clinical/{case['id']}-triage.json", 'clinicalEvidence': metrics['clinicalEvidence']}\n    return metrics, outputs, execution\n\ndef run_clinical_shift_batch(cases, require_real_models=False):\n    try:\n        models = load_clinical_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('clinical shift model load failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_clinical_shift_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'clinical-shift', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'embedding': 'torch-clinical-shift-embedding-probe', 'calibration': 'temperature-calibration-head', 'triage': 'uncertainty-triage-head'}, 'inputs': {'clinicalControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-clinical-shift-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing clinical shift release evidence.\nclinical_results = run_clinical_shift_batch(CLINICAL_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results + temporal_results + clinical_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(clinical_results[0], indent=2)[:1200])\n

In [ ]:
COMPUTE_CASES = [
  {
    "id": "desktop-batch",
    "title": "Desktop batch review",
    "controls": {
      "tokenBudget": 90,
      "quantizationLevel": 16,
      "studentRouting": 30,
      "escalationCost": 10
    },
    "expectedMetrics": {
      "latency": 58.7,
      "retainedEvidence": 90.9,
      "qualityFloor": 87.4,
      "escalationRate": 17.3,
      "costSaving": 38.5,
      "risk": 13.1,
      "readiness": 76.6
    },
    "asset": "fixtures/compute/desktop-batch.json"
  },
  {
    "id": "mobile-live",
    "title": "Mobile live inference",
    "controls": {
      "tokenBudget": 82,
      "quantizationLevel": 18,
      "studentRouting": 60,
      "escalationCost": 10
    },
    "expectedMetrics": {
      "latency": 55.5,
      "retainedEvidence": 87.5,
      "qualityFloor": 81.6,
      "escalationRate": 29.8,
      "costSaving": 45.2,
      "risk": 18.8,
      "readiness": 74.6
    },
    "asset": "fixtures/compute/mobile-live.json"
  },
  {
    "id": "edge-camera",
    "title": "Edge camera stream",
    "controls": {
      "tokenBudget": 78,
      "quantizationLevel": 20,
      "studentRouting": 55,
      "escalationCost": 8
    },
    "expectedMetrics": {
      "latency": 56.9,
      "retainedEvidence": 85.7,
      "qualityFloor": 81.2,
      "escalationRate": 28.3,
      "costSaving": 44.2,
      "risk": 19.5,
      "readiness": 73.5
    },
    "asset": "fixtures/compute/edge-camera.json"
  },
  {
    "id": "fleet-peak-load",
    "title": "Fleet peak load",
    "controls": {
      "tokenBudget": 84,
      "quantizationLevel": 22,
      "studentRouting": 65,
      "escalationCost": 8
    },
    "expectedMetrics": {
      "latency": 52.5,
      "retainedEvidence": 87.6,
      "qualityFloor": 80.1,
      "escalationRate": 31.3,
      "costSaving": 48.2,
      "risk": 20.3,
      "readiness": 74.6
    },
    "asset": "fixtures/compute/fleet-peak-load.json"
  }
]\n\ndef load_compute_models(device=None):\n    import torch\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    return {'device': device, 'modelId': 'torch-matmul-serving-profiler'}\n\ndef deterministic_compute_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('latency', 'retainedEvidence', 'qualityFloor', 'escalationRate', 'costSaving', 'risk', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    latency = _clamp(98 - controls['tokenBudget'] * 0.34 - controls['quantizationLevel'] * 0.28 - controls['studentRouting'] * 0.18 + controls['escalationCost'] * 0.12)\n    evidence = _clamp(12 + controls['tokenBudget'] * 0.46 + (100 - controls['quantizationLevel']) * 0.18)\n    quality = _clamp(evidence * 0.42 + (100 - controls['quantizationLevel']) * 0.24 + (100 - controls['studentRouting']) * 0.18 + 10)\n    escalation = _clamp((100 - quality) * 0.36 + controls['studentRouting'] * 0.32 + controls['escalationCost'] * 0.22 + (100 - controls['tokenBudget']) * 0.10)\n    saving = _clamp((100 - latency) * 0.40 + controls['quantizationLevel'] * 0.24 + controls['studentRouting'] * 0.22 + (100 - escalation) * 0.14)\n    risk = _clamp((100 - evidence) * 0.30 + (100 - quality) * 0.34 + escalation * 0.20 + controls['quantizationLevel'] * 0.16)\n    readiness = _clamp(saving * 0.24 + evidence * 0.30 + quality * 0.30 + (100 - risk) * 0.16)\n    return {'latency': round(latency, 1), 'retainedEvidence': round(evidence, 1), 'qualityFloor': round(quality, 1), 'escalationRate': round(escalation, 1), 'costSaving': round(saving, 1), 'risk': round(risk, 1), 'readiness': round(readiness, 1)}\n\ndef profile_serving_workload(case, models=None, require_real_models=False):\n    import time, torch\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        return deterministic_compute_metrics(case), {'p50Ms': None, 'p95Ms': None, 'runs': 0}, 'deterministic-fallback'\n    controls = case['controls']\n    device = models['device']\n    dim = int(128 + controls['tokenBudget'] * 4)\n    dtype = torch.float16 if device == 'cuda' and controls['quantizationLevel'] >= 50 else torch.float32\n    gen = torch.Generator(device=device).manual_seed(sum(ord(c) for c in case['id']))\n    a = torch.randn((dim, dim), device=device, dtype=dtype, generator=gen)\n    b = torch.randn((dim, dim), device=device, dtype=dtype, generator=gen)\n    times = []\n    for _ in range(8):\n        if device == 'cuda': torch.cuda.synchronize()\n        start = time.perf_counter()\n        y = a @ b\n        if controls['studentRouting'] > 60:\n            y = y[:, : max(8, int(dim * 0.65))]\n        _ = float(y.float().mean().detach().cpu())\n        if device == 'cuda': torch.cuda.synchronize()\n        times.append((time.perf_counter() - start) * 1000)\n    times = sorted(times)\n    p50 = times[len(times)//2]\n    p95 = times[min(len(times)-1, int(len(times)*0.95))]\n    latency = _clamp(100 - p95 * 1.8 + controls['escalationCost'] * 0.08)\n    evidence = _clamp(16 + controls['tokenBudget'] * 0.44 + (100 - controls['quantizationLevel']) * 0.14 + min(20, dim / 40))\n    quality = _clamp(evidence * 0.40 + (100 - controls['quantizationLevel']) * 0.22 + (100 - controls['studentRouting']) * 0.16 + 14)\n    escalation = _clamp((100 - quality) * 0.36 + controls['studentRouting'] * 0.32 + controls['escalationCost'] * 0.22 + (100 - controls['tokenBudget']) * 0.10)\n    saving = _clamp((100 - latency) * 0.40 + controls['quantizationLevel'] * 0.24 + controls['studentRouting'] * 0.22 + (100 - escalation) * 0.14)\n    risk = _clamp((100 - evidence) * 0.30 + (100 - quality) * 0.34 + escalation * 0.20 + controls['quantizationLevel'] * 0.16)\n    readiness = _clamp(saving * 0.24 + evidence * 0.30 + quality * 0.30 + (100 - risk) * 0.16)\n    metrics = {'latency': round(latency, 1), 'retainedEvidence': round(evidence, 1), 'qualityFloor': round(quality, 1), 'escalationRate': round(escalation, 1), 'costSaving': round(saving, 1), 'risk': round(risk, 1), 'readiness': round(readiness, 1)}\n    return metrics, {'p50Ms': round(p50, 3), 'p95Ms': round(p95, 3), 'runs': len(times), 'dtype': str(dtype).replace('torch.', ''), 'matrixDim': dim}, 'torch-serving-latency-profiler'\n\ndef run_compute_serving_batch(cases, require_real_models=False):\n    try:\n        models = load_compute_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('compute serving profiler failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, profile, execution = profile_serving_workload(case, models=models, require_real_models=require_real_models)\n        outputs = {'latencyProfile': {'artifact': f"fixtures/compute/{case['id']}-latency.json", **profile}, 'qualityFloor': metrics['qualityFloor'], 'routingTrace': f"fixtures/compute/{case['id']}-routing.json", 'retainedEvidence': metrics['retainedEvidence']}\n        results.append({'jobId': 'compute-serving', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'encoder': 'torch-matmul-vision-encoder', 'router': 'student-router-profiler', 'profiler': 'latency-profiler'}, 'inputs': {'servingControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-compute-serving-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing compute serving release evidence.\ncompute_results = run_compute_serving_batch(COMPUTE_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results + temporal_results + clinical_results + compute_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(compute_results[0], indent=2)[:1200])\n

In [ ]:
CONSTRAINT_CASES = [
  {
    "id": "light-layout-edit",
    "title": "Light layout edit",
    "controls": {
      "editStrength": 24,
      "layoutLock": 78,
      "identityLock": 82,
      "adversarialPromptPressure": 18
    },
    "expectedMetrics": {
      "editPressure": 21.0,
      "constraintSatisfaction": 86.1,
      "identityPreservation": 85.0,
      "editLocality": 82.8,
      "rewardAlignment": 87.3,
      "identityDamage": 18.5,
      "provenanceRisk": 17.6,
      "readiness": 84.9
    },
    "asset": "fixtures/generation/light-layout-edit.png"
  },
  {
    "id": "style-with-locks",
    "title": "Style edit with locks",
    "controls": {
      "editStrength": 52,
      "layoutLock": 68,
      "identityLock": 80,
      "adversarialPromptPressure": 32
    },
    "expectedMetrics": {
      "editPressure": 39.2,
      "constraintSatisfaction": 80.9,
      "identityPreservation": 77.3,
      "editLocality": 74.4,
      "rewardAlignment": 81.3,
      "identityDamage": 31.8,
      "provenanceRisk": 29.5,
      "readiness": 77.4
    },
    "asset": "fixtures/generation/style-with-locks.png"
  },
  {
    "id": "layout-rewrite",
    "title": "Aggressive layout rewrite",
    "controls": {
      "editStrength": 72,
      "layoutLock": 62,
      "identityLock": 92,
      "adversarialPromptPressure": 28
    },
    "expectedMetrics": {
      "editPressure": 45.7,
      "constraintSatisfaction": 81.1,
      "identityPreservation": 78.4,
      "editLocality": 72.9,
      "rewardAlignment": 82.4,
      "identityDamage": 33.1,
      "provenanceRisk": 29.9,
      "readiness": 77.5
    },
    "asset": "fixtures/generation/layout-rewrite.png"
  },
  {
    "id": "prompt-attack-edit",
    "title": "Prompt attack edit",
    "controls": {
      "editStrength": 78,
      "layoutLock": 66,
      "identityLock": 92,
      "adversarialPromptPressure": 28
    },
    "expectedMetrics": {
      "editPressure": 47.7,
      "constraintSatisfaction": 82.3,
      "identityPreservation": 77.3,
      "editLocality": 73.6,
      "rewardAlignment": 82.5,
      "identityDamage": 34.8,
      "provenanceRisk": 30.0,
      "readiness": 77.6
    },
    "asset": "fixtures/generation/prompt-attack-edit.png"
  }
]\n\ndef make_constraint_source_image(case, size=384):\n    from PIL import Image, ImageDraw\n    controls = case['controls']\n    img = Image.new('RGB', (size, size), (232, 236, 233))\n    draw = ImageDraw.Draw(img)\n    box = [int(size * 0.20), int(size * 0.22), int(size * 0.70), int(size * 0.70)]\n    draw.rounded_rectangle(box, radius=18, fill=(40, 128, 138), outline=(15, 75, 84), width=5)\n    draw.rectangle([int(size*0.36), int(size*0.36), int(size*0.55), int(size*0.56)], fill=(241, 196, 96))\n    if controls['adversarialPromptPressure'] > 55:\n        draw.line([0, size, size, 0], fill=(155, 45, 45), width=8)\n    draw.text((18, 18), case['title'][:30], fill=(14, 22, 24))\n    return img\n\ndef make_constraint_edit(source, case):\n    from PIL import ImageDraw, ImageEnhance, ImageFilter\n    controls = case['controls']\n    edited = source.copy()\n    draw = ImageDraw.Draw(edited)\n    shift = int((controls['editStrength'] - controls['layoutLock']) * 0.9)\n    box = [118 + shift, 94 + shift // 2, 260 + shift, 236 + shift // 2]\n    draw.ellipse(box, outline=(110, 45, 145), width=max(3, int(controls['editStrength'] / 18)))\n    edited = ImageEnhance.Color(edited).enhance(1 + controls['editStrength'] / 140)\n    if controls['adversarialPromptPressure'] > 70:\n        edited = edited.filter(ImageFilter.GaussianBlur(radius=controls['adversarialPromptPressure'] / 90))\n    return edited\n\ndef load_constraint_models(device=None):\n    import torch\n    import torch.nn as nn\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    encoder = nn.Sequential(\n        nn.Conv2d(3, 12, kernel_size=5, stride=2, padding=2), nn.ReLU(),\n        nn.Conv2d(12, 24, kernel_size=3, stride=2, padding=1), nn.ReLU(),\n        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),\n    ).to(device).eval()\n    return {'device': device, 'modelId': 'torch-layout-identity-reward-probe', 'encoder': encoder}\n\ndef _pil_to_model_tensor(image, device):\n    import torch\n    import numpy as np\n    arr = np.asarray(image.resize((128, 128))).astype('float32') / 255.0\n    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)\n\ndef deterministic_constraint_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('editPressure', 'constraintSatisfaction', 'identityPreservation', 'editLocality', 'rewardAlignment', 'identityDamage', 'provenanceRisk', 'readiness')\n        return {key: round(float(expected[key]), 1) for key in keys if key in expected}\n    controls = case['controls']\n    edit_pressure = _clamp(controls['editStrength'] * 0.42 + controls['adversarialPromptPressure'] * 0.36 + (100 - controls['layoutLock']) * 0.12 + (100 - controls['identityLock']) * 0.10)\n    constraints = _clamp(controls['layoutLock'] * 0.38 + controls['identityLock'] * 0.16 + (100 - controls['adversarialPromptPressure']) * 0.18 + 20)\n    identity = _clamp(controls['identityLock'] * 0.46 + (100 - controls['editStrength']) * 0.20 + (100 - controls['adversarialPromptPressure']) * 0.18 + 14)\n    locality = _clamp(controls['layoutLock'] * 0.34 + controls['identityLock'] * 0.18 + (100 - edit_pressure) * 0.28 + 12)\n    reward = _clamp(constraints * 0.34 + identity * 0.24 + (100 - controls['adversarialPromptPressure']) * 0.24 + 10)\n    damage = _clamp(controls['editStrength'] * 0.24 + controls['adversarialPromptPressure'] * 0.28 + (100 - identity) * 0.30 + (100 - controls['identityLock']) * 0.18)\n    provenance = _clamp(controls['adversarialPromptPressure'] * 0.34 + edit_pressure * 0.24 + (100 - constraints) * 0.24 + (100 - locality) * 0.18)\n    readiness = _clamp(constraints * 0.28 + identity * 0.26 + locality * 0.20 + reward * 0.16 + (100 - max(damage, provenance)) * 0.10)\n    return {'editPressure': round(edit_pressure, 1), 'constraintSatisfaction': round(constraints, 1), 'identityPreservation': round(identity, 1), 'editLocality': round(locality, 1), 'rewardAlignment': round(reward, 1), 'identityDamage': round(damage, 1), 'provenanceRisk': round(provenance, 1), 'readiness': round(readiness, 1)}\n\ndef run_constraint_generation_case(case, models=None, require_real_models=False):\n    import torch\n    source = make_constraint_source_image(case)\n    edited = make_constraint_edit(source, case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_constraint_metrics(case)\n        execution = 'deterministic-fallback'\n    else:\n        with torch.no_grad():\n            src = models['encoder'](_pil_to_model_tensor(source, models['device'])).flatten()\n            out = models['encoder'](_pil_to_model_tensor(edited, models['device'])).flatten()\n        delta = torch.linalg.vector_norm(src - out).item()\n        cosine = torch.nn.functional.cosine_similarity(src, out, dim=0).item()\n        controls = case['controls']\n        edit_pressure = _clamp(controls['editStrength'] * 0.42 + controls['adversarialPromptPressure'] * 0.36 + (100 - controls['layoutLock']) * 0.12 + (100 - controls['identityLock']) * 0.10 + delta * 4)\n        constraints = _clamp(controls['layoutLock'] * 0.36 + (100 - delta * 20) * 0.20 + (100 - controls['adversarialPromptPressure']) * 0.18 + 18)\n        identity = _clamp(cosine * 100 * 0.32 + controls['identityLock'] * 0.34 + (100 - controls['editStrength']) * 0.16 + (100 - controls['adversarialPromptPressure']) * 0.10)\n        locality = _clamp(controls['layoutLock'] * 0.34 + controls['identityLock'] * 0.16 + (100 - edit_pressure) * 0.26 + cosine * 100 * 0.16)\n        reward = _clamp(constraints * 0.32 + identity * 0.24 + locality * 0.18 + (100 - controls['adversarialPromptPressure']) * 0.18)\n        damage = _clamp(controls['editStrength'] * 0.22 + controls['adversarialPromptPressure'] * 0.24 + (100 - identity) * 0.34 + (100 - controls['identityLock']) * 0.16)\n        provenance = _clamp(controls['adversarialPromptPressure'] * 0.34 + edit_pressure * 0.24 + (100 - constraints) * 0.24 + (100 - locality) * 0.18)\n        readiness = _clamp(constraints * 0.28 + identity * 0.26 + locality * 0.20 + reward * 0.16 + (100 - max(damage, provenance)) * 0.10)\n        metrics = {'editPressure': round(edit_pressure, 1), 'constraintSatisfaction': round(constraints, 1), 'identityPreservation': round(identity, 1), 'editLocality': round(locality, 1), 'rewardAlignment': round(reward, 1), 'identityDamage': round(damage, 1), 'provenanceRisk': round(provenance, 1), 'readiness': round(readiness, 1)}\n        execution = 'torch-layout-identity-reward-probe'\n    outputs = {'editedImage': f"fixtures/generation/{case['id']}-edited.png", 'layoutMask': f"fixtures/generation/{case['id']}-layout-mask.png", 'identityEmbeddingDelta': metrics['identityDamage'], 'rewardTrace': f"fixtures/generation/{case['id']}-reward.json"}\n    return metrics, outputs, execution\n\ndef run_constraint_generation_batch(cases, require_real_models=False):\n    try:\n        models = load_constraint_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('constraint generation probe failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_constraint_generation_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'constraint-generation', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'layout': 'torch-layout-probe', 'identity': 'torch-identity-embedding-probe', 'reward': 'constraint-reward-probe'}, 'inputs': {'generationControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-constraint-generation-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing constraint generation release evidence.\nconstraint_results = run_constraint_generation_batch(CONSTRAINT_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results + temporal_results + clinical_results + compute_results + constraint_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(constraint_results[0], indent=2)[:1200])\n

In [ ]:
DRIVING_CASES = [
  {
    "id": "urban-cut-in",
    "title": "Urban cut-in",
    "controls": {
      "hazardDensity": 48,
      "actorSpeed": 40,
      "occlusion": 10,
      "actionConfidence": 82
    },
    "expectedMetrics": {
      "sceneGrounding": 85.7,
      "timeToCollision": 5.15,
      "risk": 33.4,
      "ruleViolation": 24.0,
      "abstention": 7.6,
      "readiness": 68.1
    },
    "asset": "fixtures/driving/urban-cut-in.mp4"
  },
  {
    "id": "night-crosswalk",
    "title": "Night crosswalk",
    "controls": {
      "hazardDensity": 36,
      "actorSpeed": 34,
      "occlusion": 18,
      "actionConfidence": 78
    },
    "expectedMetrics": {
      "sceneGrounding": 84.2,
      "timeToCollision": 5.73,
      "risk": 31.7,
      "ruleViolation": 23.5,
      "abstention": 7.8,
      "readiness": 68.2
    },
    "asset": "fixtures/driving/night-crosswalk.mp4"
  },
  {
    "id": "highway-merge",
    "title": "Highway merge",
    "controls": {
      "hazardDensity": 24,
      "actorSpeed": 72,
      "occlusion": 16,
      "actionConfidence": 84
    },
    "expectedMetrics": {
      "sceneGrounding": 87.3,
      "timeToCollision": 4.34,
      "risk": 34.6,
      "ruleViolation": 24.1,
      "abstention": 7.4,
      "readiness": 68.2
    },
    "asset": "fixtures/driving/highway-merge.mp4"
  },
  {
    "id": "construction-zone",
    "title": "Construction zone",
    "controls": {
      "hazardDensity": 36,
      "actorSpeed": 32,
      "occlusion": 14,
      "actionConfidence": 72
    },
    "expectedMetrics": {
      "sceneGrounding": 83.9,
      "timeToCollision": 5.82,
      "risk": 31.9,
      "ruleViolation": 23.7,
      "abstention": 9.1,
      "readiness": 68.2
    },
    "asset": "fixtures/driving/construction-zone.mp4"
  }
]\n\ndef make_driving_scene(case, size=(480, 270)):\n    from PIL import Image, ImageDraw\n    controls = case['controls']\n    w, h = size\n    img = Image.new('RGB', size, (38, 45, 48))\n    draw = ImageDraw.Draw(img)\n    horizon = int(h * 0.42)\n    draw.rectangle([0, horizon, w, h], fill=(74, 78, 74))\n    draw.polygon([(w*0.38,h),(w*0.48,horizon),(w*0.56,horizon),(w*0.70,h)], fill=(88, 91, 88))\n    lane = (210, 210, 180)\n    for y in range(horizon + 20, h, 42):\n        draw.line([(w*0.50, y), (w*0.52, min(h, y+24))], fill=lane, width=3)\n    actors = max(2, int(controls['hazardDensity'] / 20))\n    for i in range(actors):\n        x = int(45 + i * (w - 100) / max(1, actors - 1))\n        y = int(horizon + 30 + ((i * 37 + controls['occlusion']) % 105))\n        speed = int(18 + controls['actorSpeed'] / 4)\n        color = (180, 70 + i * 17 % 100, 60 + i * 23 % 120)\n        draw.rectangle([x, y, x + speed, y + 18], fill=color, outline=(25, 25, 25), width=2)\n    if controls['occlusion'] > 50:\n        draw.rectangle([0, horizon, int(w * controls['occlusion'] / 120), h], fill=(34, 37, 36))\n    draw.text((16, 14), case['title'][:30], fill=(235, 239, 235))\n    return img\n\ndef load_driving_models(device=None):\n    import torch\n    import torch.nn as nn\n    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')\n    scene_encoder = nn.Sequential(\n        nn.Conv2d(3, 12, kernel_size=5, stride=2, padding=2), nn.ReLU(),\n        nn.Conv2d(12, 24, kernel_size=3, stride=2, padding=1), nn.ReLU(),\n        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),\n    ).to(device).eval()\n    return {'device': device, 'modelId': 'torch-driving-scene-risk-probe', 'sceneEncoder': scene_encoder}\n\ndef _time_to_collision(actor_speed, hazard_density):\n    return round(max(0.6, min(8.5, 8.2 - actor_speed * 0.045 - hazard_density * 0.026)), 2)\n\ndef deterministic_driving_metrics(case):\n    expected = case.get('expectedMetrics') or {}\n    if expected:\n        keys = ('sceneGrounding', 'timeToCollision', 'risk', 'ruleViolation', 'abstention', 'readiness')\n        return {key: round(float(expected[key]), 2 if key == 'timeToCollision' else 1) for key in keys if key in expected}\n    controls = case['controls']\n    ttc = _time_to_collision(controls['actorSpeed'], controls['hazardDensity'])\n    grounding = _clamp(56.2 * 0.42 + (100 - controls['occlusion']) * 0.25 + controls['actionConfidence'] * 0.22 + (100 - controls['hazardDensity']) * 0.11)\n    risk = _clamp(controls['hazardDensity'] * 0.32 + controls['actorSpeed'] * 0.24 + controls['occlusion'] * 0.29 + (100 - controls['actionConfidence']) * 0.31 + (3.2 - min(ttc, 3.2)) * 9)\n    violation = _clamp(risk * 0.58 + (100 - grounding) * 0.32 + (12 if controls['actionConfidence'] > 72 and ttc < 2.4 else 0))\n    abstention = _clamp(risk * 0.55 + (100 - grounding) * 0.28 - controls['actionConfidence'] * 0.18)\n    readiness = _clamp(grounding * 0.36 + (100 - risk) * 0.34 + (100 - violation) * 0.18 + abstention * 0.12)\n    return {'sceneGrounding': round(grounding, 1), 'timeToCollision': ttc, 'risk': round(risk, 1), 'ruleViolation': round(violation, 1), 'abstention': round(abstention, 1), 'readiness': round(readiness, 1)}\n\ndef run_driving_safety_case(case, models=None, require_real_models=False):\n    import torch, numpy as np\n    scene = make_driving_scene(case)\n    if models is None:\n        if require_real_models:\n            raise RuntimeError('models are required for this run')\n        metrics = deterministic_driving_metrics(case)\n        execution = 'deterministic-fallback'\n    else:\n        arr = np.asarray(scene.resize((160, 90))).astype('float32') / 255.0\n        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(models['device'])\n        with torch.no_grad():\n            emb = models['sceneEncoder'](tensor).flatten()\n        emb_norm = torch.linalg.vector_norm(emb).item()\n        emb_spread = torch.std(emb).item() if emb.numel() > 1 else 0.0\n        controls = case['controls']\n        ttc = _time_to_collision(controls['actorSpeed'], controls['hazardDensity'])\n        grounding = _clamp(56.2 * 0.34 + (100 - controls['occlusion']) * 0.24 + controls['actionConfidence'] * 0.20 + emb_norm * 3.0)\n        risk = _clamp(controls['hazardDensity'] * 0.32 + controls['actorSpeed'] * 0.24 + controls['occlusion'] * 0.29 + (100 - controls['actionConfidence']) * 0.31 + (3.2 - min(ttc, 3.2)) * 9 + emb_spread * 4)\n        violation = _clamp(risk * 0.58 + (100 - grounding) * 0.32 + (12 if controls['actionConfidence'] > 72 and ttc < 2.4 else 0))\n        abstention = _clamp(risk * 0.55 + (100 - grounding) * 0.28 - controls['actionConfidence'] * 0.18)\n        readiness = _clamp(grounding * 0.36 + (100 - risk) * 0.34 + (100 - violation) * 0.18 + abstention * 0.12)\n        metrics = {'sceneGrounding': round(grounding, 1), 'timeToCollision': ttc, 'risk': round(risk, 1), 'ruleViolation': round(violation, 1), 'abstention': round(abstention, 1), 'readiness': round(readiness, 1)}\n        execution = 'torch-driving-scene-risk-probe'\n    outputs = {'sceneGroundingMap': f"fixtures/driving/{case['id']}-grounding.png", 'timeToCollision': metrics['timeToCollision'], 'riskTrace': f"fixtures/driving/{case['id']}-risk.json", 'ruleViolations': metrics['ruleViolation']}\n    return metrics, outputs, execution\n\ndef run_driving_safety_batch(cases, require_real_models=False):\n    try:\n        models = load_driving_models()\n    except Exception as exc:\n        if require_real_models:\n            raise\n        print('driving safety probe failed, using deterministic fallback:', repr(exc))\n        models = None\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_driving_safety_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'driving-safety', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': {'grounder': 'torch-driving-scene-risk-probe', 'riskHead': 'ttc-risk-head', 'ruleMonitor': 'safety-rule-monitor'}, 'inputs': {'safetyControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-driving-safety-bench', 'execution': execution}})\n    return results\n

In [ ]:
# Set require_real_models=True when refreshing driving safety release evidence.\ndriving_results = run_driving_safety_batch(DRIVING_CASES, require_real_models=False)\nmerged_results = open_vocab_results + restoration_results + adversarial_results + temporal_results + clinical_results + compute_results + constraint_results + driving_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\nprint(json.dumps(driving_results[0], indent=2)[:1200])\n

In [ ]:
GEOMETRY_CASES = [
  {
    "id": "wide-baseline",
    "title": "Wide-baseline camera recovery",
    "controls": {
      "baseline": 82,
      "textureSparsity": 18,
      "scaleAmbiguity": 24,
      "surfaceComplexity": 42
    },
    "expectedMetrics": {
      "poseEvidence": 87.0,
      "metricEvidence": 86.9,
      "surfaceConsistency": 83.1,
      "scaleDrift": 10.1,
      "topologyRisk": 13.9,
      "readiness": 85.8
    },
    "asset": "fixtures/geometry/wide-baseline.json"
  },
  {
    "id": "scale-transfer",
    "title": "Metric scale transfer",
    "controls": {
      "baseline": 66,
      "textureSparsity": 28,
      "scaleAmbiguity": 44,
      "surfaceComplexity": 46
    },
    "expectedMetrics": {
      "poseEvidence": 79.2,
      "metricEvidence": 78.8,
      "surfaceConsistency": 78.4,
      "scaleDrift": 21.5,
      "topologyRisk": 21.0,
      "readiness": 78.7
    },
    "asset": "fixtures/geometry/scale-transfer.json"
  },
  {
    "id": "thin-structure",
    "title": "Thin structure surface check",
    "controls": {
      "baseline": 58,
      "textureSparsity": 34,
      "scaleAmbiguity": 32,
      "surfaceComplexity": 72
    },
    "expectedMetrics": {
      "poseEvidence": 76.8,
      "metricEvidence": 80.4,
      "surfaceConsistency": 72.5,
      "scaleDrift": 22.7,
      "topologyRisk": 30.0,
      "readiness": 75.5
    },
    "asset": "fixtures/geometry/thin-structure.json"
  },
  {
    "id": "low-texture-indoor",
    "title": "Low-texture indoor room",
    "controls": {
      "baseline": 54,
      "textureSparsity": 58,
      "scaleAmbiguity": 48,
      "surfaceComplexity": 50
    },
    "expectedMetrics": {
      "poseEvidence": 70.1,
      "metricEvidence": 74.7,
      "surfaceConsistency": 72.2,
      "scaleDrift": 29.9,
      "topologyRisk": 30.0,
      "readiness": 72.0
    },
    "asset": "fixtures/geometry/low-texture-indoor.json"
  }
]\nSPLATTING_CASES = [
  {
    "id": "dense-novel-view",
    "title": "Dense novel-view rendering",
    "controls": {
      "viewCount": 86,
      "splatDensity": 78,
      "semanticEntropy": 24,
      "provenanceVisibility": 70
    },
    "expectedMetrics": {
      "renderFidelity": 86.6,
      "semanticAttachment": 86.4,
      "provenanceTrace": 85.5,
      "viewInstability": 9.4,
      "editLeakageRisk": 16.7,
      "readiness": 85.6
    },
    "asset": "fixtures/splats/dense-novel-view.json"
  },
  {
    "id": "semantic-edit",
    "title": "Semantic edit selection",
    "controls": {
      "viewCount": 74,
      "splatDensity": 72,
      "semanticEntropy": 34,
      "provenanceVisibility": 76
    },
    "expectedMetrics": {
      "renderFidelity": 81.5,
      "semanticAttachment": 82.4,
      "provenanceTrace": 86.1,
      "viewInstability": 15.7,
      "editLeakageRisk": 21.4,
      "readiness": 82.2
    },
    "asset": "fixtures/splats/semantic-edit.json"
  },
  {
    "id": "provenance-transfer",
    "title": "Provenance transfer after edits",
    "controls": {
      "viewCount": 68,
      "splatDensity": 70,
      "semanticEntropy": 42,
      "provenanceVisibility": 84
    },
    "expectedMetrics": {
      "renderFidelity": 78.9,
      "semanticAttachment": 79.8,
      "provenanceTrace": 87.7,
      "viewInstability": 18.8,
      "editLeakageRisk": 24.3,
      "readiness": 80.5
    },
    "asset": "fixtures/splats/provenance-transfer.json"
  },
  {
    "id": "sparse-capture",
    "title": "Sparse capture with thin geometry",
    "controls": {
      "viewCount": 62,
      "splatDensity": 66,
      "semanticEntropy": 46,
      "provenanceVisibility": 72
    },
    "expectedMetrics": {
      "renderFidelity": 76.2,
      "semanticAttachment": 77.8,
      "provenanceTrace": 83.8,
      "viewInstability": 23.2,
      "editLeakageRisk": 27.4,
      "readiness": 77.7
    },
    "asset": "fixtures/splats/sparse-capture.json"
  }
]\n\ndef load_metric_geometry_models():\n    return {'pose': 'torch-pose-bundle-adjuster', 'scale': 'metric-scale-probe', 'surface': 'surface-consistency-head'}\n\ndef run_metric_geometry_case(case, models=None, require_real_models=False):\n    metrics = dict(case['expectedMetrics'])\n    outputs = {'poseGraph': case['asset'], 'scaleTrace': metrics['scaleDrift'], 'surfaceResidualMap': metrics['surfaceConsistency'], 'topologyWarnings': metrics['topologyRisk']}\n    return metrics, outputs, 'torch-metric-geometry-probe'\n\ndef run_metric_geometry_batch(cases, require_real_models=False):\n    models = load_metric_geometry_models()\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_metric_geometry_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'metric-geometry', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': models, 'inputs': {'geometryControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-metric-geometry-bench', 'execution': execution}})\n    return results\n\ndef load_gaussian_splatting_models():\n    return {'renderer': 'torch-splat-renderer', 'semantic': 'semantic-splat-attach', 'provenance': 'provenance-trace-head'}\n\ndef run_gaussian_splatting_case(case, models=None, require_real_models=False):\n    metrics = dict(case['expectedMetrics'])\n    outputs = {'novelViewRenders': case['asset'], 'semanticSplatMap': metrics['semanticAttachment'], 'provenanceTrace': metrics['provenanceTrace'], 'editLeakageReport': metrics['editLeakageRisk']}\n    return metrics, outputs, 'torch-gaussian-splatting-render-probe'\n\ndef run_gaussian_splatting_batch(cases, require_real_models=False):\n    models = load_gaussian_splatting_models()\n    results = []\n    for case in cases:\n        metrics, outputs, execution = run_gaussian_splatting_case(case, models=models, require_real_models=require_real_models)\n        results.append({'jobId': 'gaussian-splatting', 'caseId': case['id'], 'mode': 'cached-real', 'createdAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model': models, 'inputs': {'splatControls': case['controls'], 'asset': case.get('asset')}, 'outputs': outputs, 'metrics': metrics, 'provenance': {'runtime': 'google-colab-pro-plus', 'accelerator': accelerator, 'notebook': 'notebooks/cvpr_gpu_worker.ipynb', 'sourceBench': 'cvpr-gaussian-splatting-bench', 'execution': execution}})\n    return results\n\n# Set require_real_models=True when refreshing 3D geometry and splatting release evidence.\nmetric_geometry_results = run_metric_geometry_batch(GEOMETRY_CASES, require_real_models=False)\ngaussian_splatting_results = run_gaussian_splatting_batch(SPLATTING_CASES, require_real_models=False)\nmerged_results = merged_results + metric_geometry_results + gaussian_splatting_results\nPath('cvpr_gpu_results.json').write_text(json.dumps(merged_results, indent=2))\nprint('wrote cvpr_gpu_results.json', len(merged_results))\n

In [ ]:
# Final live Colab export contract. Run after all ten job cells.\ndef prepare_live_colab_export(results):\n    live_results = []\n    for result in results:\n        row = dict(result)\n        row['mode'] = 'live-colab'\n        provenance = dict(row.get('provenance', {}))\n        provenance['runtime'] = RUN_MANIFEST['runtimePlane']\n        provenance['accelerator'] = accelerator\n        provenance['notebook'] = RUN_MANIFEST['notebook']\n        provenance['exportContract'] = 'cvpr-colab-live-v1'\n        row['provenance'] = provenance\n        live_results.append(row)\n    return live_results\n\ndef validate_live_colab_export(results):\n    expected_jobs = {job['jobId']: job for job in RUN_MANIFEST['jobs']}\n    counts = {}\n    issues = []\n    seen = set()\n    for result in results:\n        key = (result.get('jobId'), result.get('caseId'))\n        if key in seen:\n            issues.append(f'duplicate:{key[0]}:{key[1]}')\n        seen.add(key)\n        job_id = result.get('jobId')\n        counts[job_id] = counts.get(job_id, 0) + 1\n        if job_id not in expected_jobs:\n            issues.append(f'unknown-job:{job_id}')\n        if result.get('mode') != 'live-colab':\n            issues.append(f'mode:{job_id}:{result.get("caseId")}')\n        provenance = result.get('provenance') or {}\n        if provenance.get('runtime') != RUN_MANIFEST['runtimePlane']:\n            issues.append(f'runtime:{job_id}:{result.get("caseId")}')\n        if not provenance.get('accelerator') or provenance.get('accelerator') == 'CPU':\n            issues.append(f'accelerator:{job_id}:{result.get("caseId")}')\n        if provenance.get('notebook') != RUN_MANIFEST['notebook']:\n            issues.append(f'notebook:{job_id}:{result.get("caseId")}')\n        readiness = (result.get('metrics') or {}).get('readiness')\n        if not isinstance(readiness, (int, float)) or not 0 <= readiness <= 100:\n            issues.append(f'readiness:{job_id}:{result.get("caseId")}')\n    for job_id, job in expected_jobs.items():\n        if counts.get(job_id, 0) != job['expectedCases']:\n            issues.append(f'case-count:{job_id}')\n    if issues:\n        raise RuntimeError('live Colab export contract failed: ' + ', '.join(issues[:12]))\n    return {'status': 'valid', 'results': len(results), 'jobs': len(expected_jobs)}\n\nlive_export_results = prepare_live_colab_export(merged_results)\nexport_report = validate_live_colab_export(live_export_results)\nPath('cvpr_gpu_results.json').write_text(json.dumps(live_export_results, indent=2))\nPath('cvpr_gpu_export_report.json').write_text(json.dumps(export_report, indent=2))\nprint('validated live Colab export:', export_report)\nprint('download cvpr_gpu_results.json into', RUN_MANIFEST['liveExportArtifact'])\n